# Phase 5 (Single-Pass) — Resolution Sensitivity Analysis

Repeats the Phase 5 resolution sweep **without** sliding-window chunking.
Every resolution is evaluated in a single VGGT forward pass over the same
8 frames, eliminating Procrustes stitching error as a confound.

**Why 8 frames?**  
A single VGGT call on a T4 (14.6 GB) peaks at ~9.5 GB at 518 px with 8
frames — the tightest budget point.  Using 8 frames keeps every resolution
in a single pass safely within VRAM.  Reduce `N_FRAMES` to 6 if OOM errors
appear.

**Comparison with chunked run (notebook 04)**  
| | Chunked (04) | Single-pass (this) |
|---|---|---|
| Frames | 40 | 8 |
| Chunks | 7 | 1 |
| Stitching error | yes | **no** |
| ATE dominated by | chunk drift | per-frame quality |

**Expected outcome**  
If resolution genuinely affects VGGT accuracy, ATE should now vary with
resolution.  Flat ATE here would indicate the model is robust to the
224–518 px range even without stitching artifacts.

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

## 1 — Imports

In [ ]:
import os, sys
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi           import TUMVIDataset
from src.resolution_sweep import ResolutionSweeper, DEFAULT_RESOLUTIONS
from src.metrics          import compute_ate, compute_rpe, print_metrics_table

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2 — Download TUM-VI `room1`

In [ ]:
SEQUENCE     = "room1"
# 8 frames = one chunk from the previous experiment; proven to fit on T4 at 518 px.
# Lower to 6 if you hit OOM.
N_FRAMES     = 8
DOWNLOAD_DIR = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_dir        = data["image_dir"]
image_timestamps = data["image_timestamps"]
gt_extrinsics    = data["gt_extrinsics"]   # (N, 3, 4)

N = len(gt_extrinsics)
print(f"Loaded {N} frames   GT shape: {gt_extrinsics.shape}")

## 3 — Load Model

In [ ]:
# chunk_size=None (default) → single-pass VGGT, no sliding window, no Procrustes stitching.
sweeper = ResolutionSweeper(
    resolutions      = DEFAULT_RESOLUTIONS,   # [224, 280, 336, 392, 448, 518]
    conf_thresh      = 5.0,
    store_extrinsics = False,
    # chunk_size intentionally omitted → single-pass mode
)
sweeper.load_model()

## 4 — Run Resolution Sweep (single-pass)

In [ ]:
results = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = True,
    with_scale    = True,
)

df = sweeper.to_dataframe(results)
print("\nResolution sweep results (single-pass):")
print(df.to_string(index=False, float_format="{:.4f}".format))

## 5 — Time & Memory vs Resolution

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Inference Time (s)", "Peak GPU Memory (MB)"))

fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["time_s"],
               mode="lines+markers", name="Time (s)",
               line=dict(color="steelblue", width=2), marker=dict(size=8)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["peak_mb"],
               mode="lines+markers", name="Peak MB",
               line=dict(color="darkorange", width=2), marker=dict(size=8)),
    row=1, col=2,
)
fig.update_xaxes(title_text="Resolution (px)")
fig.update_layout(height=400,
                  title_text="Compute cost vs Input Resolution (single-pass, 8 frames)",
                  showlegend=False)
fig.show()

## 6 — ATE vs Resolution

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_mean"],
    mode="lines+markers", name="ATE mean",
    line=dict(color="royalblue", width=2), marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_rmse"],
    mode="lines+markers", name="ATE RMSE",
    line=dict(color="royalblue", width=2, dash="dash"), marker=dict(size=6),
))
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_median"],
    mode="lines+markers", name="ATE median",
    line=dict(color="royalblue", width=2, dash="dot"), marker=dict(size=6),
))

fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")

fig.update_layout(
    title=f"ATE vs Input Resolution — TUM-VI {SEQUENCE} ({N} frames, single-pass)",
    xaxis_title="Resolution (px)",
    yaxis_title="ATE (m, Umeyama-aligned)",
    legend=dict(x=0.6, y=0.95),
)
fig.show()

## 7 — RPE vs Resolution

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("RPE Translation (m)", "RPE Rotation (deg)"))

fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["rpe_trans"],
               mode="lines+markers", line=dict(color="forestgreen", width=2),
               marker=dict(size=8), name="RPE trans"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["rpe_rot"],
               mode="lines+markers", line=dict(color="firebrick", width=2),
               marker=dict(size=8), name="RPE rot"),
    row=1, col=2,
)
fig.update_xaxes(title_text="Resolution (px)")
fig.update_layout(height=400,
                  title_text="Relative Pose Error vs Resolution (single-pass)",
                  showlegend=False)
fig.show()

## 8 — Rotation Error vs Resolution

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["rot_mean_deg"],
    mode="lines+markers", name="Mean rotation error (deg)",
    line=dict(color="purple", width=2), marker=dict(size=8),
))
fig.update_layout(
    title="Mean Rotation Error vs Input Resolution (single-pass)",
    xaxis_title="Resolution (px)",
    yaxis_title="Mean rotation error (deg)",
)
fig.show()

## 9 — Depth Confidence vs Resolution

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["depth_conf_mean"],
    mode="lines+markers", name="Mean depth confidence",
    line=dict(color="teal", width=2), marker=dict(size=8),
))
fig.update_layout(
    title="Mean Depth Confidence vs Resolution (single-pass)",
    xaxis_title="Resolution (px)",
    yaxis_title="Mean confidence",
)
fig.show()

## 10 — Accuracy / Efficiency Trade-off

In [ ]:
fig = px.scatter(
    df, x="time_s", y="ate_mean",
    text="resolution", color="resolution",
    color_continuous_scale="Viridis",
    title="Accuracy-Efficiency Trade-off (lower-left = better)",
    labels={"time_s": "Inference time (s)", "ate_mean": "ATE mean (m)"},
    size=[10] * len(df),
)
fig.update_traces(textposition="top center")
fig.show()

## 11 — Summary Table & Save

In [ ]:
display_cols = [
    "resolution", "ate_mean", "ate_rmse",
    "rpe_trans", "rpe_rot", "rot_mean_deg",
    "time_s", "peak_mb",
]

print(f"\n=== Resolution Sensitivity (single-pass) -- TUM-VI {SEQUENCE} ({N} frames) ===")
print(df[display_cols].to_string(index=False, float_format="{:.4f}".format))

os.makedirs("results", exist_ok=True)
df.to_csv(f"results/phase5b_resolution_sweep_singlepass_{SEQUENCE}.csv", index=False)
print(f"\nSaved to results/phase5b_resolution_sweep_singlepass_{SEQUENCE}.csv")

## 12 — Side-by-side Comparison with Chunked Run

Load the Phase 5 chunked results (if present) and plot ATE side-by-side.

In [ ]:
chunked_csv = f"results/phase5_resolution_sweep_{SEQUENCE}.csv"

if os.path.isfile(chunked_csv):
    df_chunked = pd.read_csv(chunked_csv)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_chunked["resolution"], y=df_chunked["ate_mean"],
        mode="lines+markers", name="Chunked (40 frames)",
        line=dict(color="tomato", width=2, dash="dash"), marker=dict(size=8),
    ))
    fig.add_trace(go.Scatter(
        x=df["resolution"], y=df["ate_mean"],
        mode="lines+markers", name=f"Single-pass ({N} frames)",
        line=dict(color="royalblue", width=2), marker=dict(size=8),
    ))
    fig.add_vline(x=518, line_dash="dash", line_color="grey",
                  annotation_text="Native 518 px", annotation_position="top left")
    fig.update_layout(
        title="ATE vs Resolution: Chunked vs Single-Pass",
        xaxis_title="Resolution (px)",
        yaxis_title="ATE mean (m, Umeyama-aligned)",
        legend=dict(x=0.6, y=0.95),
    )
    fig.show()

    # RPE translation comparison
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=df_chunked["resolution"], y=df_chunked["rpe_trans"],
        mode="lines+markers", name="Chunked RPE-t",
        line=dict(color="tomato", width=2, dash="dash"), marker=dict(size=8),
    ))
    fig2.add_trace(go.Scatter(
        x=df["resolution"], y=df["rpe_trans"],
        mode="lines+markers", name="Single-pass RPE-t",
        line=dict(color="forestgreen", width=2), marker=dict(size=8),
    ))
    fig2.update_layout(
        title="RPE Translation vs Resolution: Chunked vs Single-Pass",
        xaxis_title="Resolution (px)",
        yaxis_title="RPE translation (m)",
        legend=dict(x=0.6, y=0.95),
    )
    fig2.show()
else:
    print(f"Chunked CSV not found at {chunked_csv} — skipping comparison plot.")
    print("Run notebook 04_resolution_sensitivity.ipynb first to generate it.")

## 13 — P2: Unaligned ATE
Does Umeyama alignment absorb real resolution variation? Re-run with `align=False`
so errors are not normalised by a per-resolution similarity transform.

In [ ]:
results_noalign = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = False,
    with_scale    = False,
)
df_noalign = sweeper.to_dataframe(results_noalign)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_mean"],
    mode="lines+markers", name="Aligned ATE (Umeyama+scale)",
    line=dict(color="royalblue", width=2), marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=df_noalign["resolution"], y=df_noalign["ate_mean"],
    mode="lines+markers", name="Unaligned ATE",
    line=dict(color="tomato", width=2, dash="dash"), marker=dict(size=8),
))
fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")
fig.update_layout(
    title=f"Aligned vs Unaligned ATE — TUM-VI {SEQUENCE} ({N} frames)",
    xaxis_title="Resolution (px)",
    yaxis_title="ATE mean (m)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

print("\nAligned ATE:")
print(df[["resolution", "ate_mean", "ate_rmse"]].to_string(index=False, float_format="{:.4f}".format))
print("\nUnaligned ATE:")
print(df_noalign[["resolution", "ate_mean", "ate_rmse"]].to_string(index=False, float_format="{:.4f}".format))

## 14 — P3: Extended Resolution Sweep (112–518 px)
The 224–518 px range tested so far may all be above VGGT's accuracy floor.
Extending to 112 px and 168 px locates where accuracy first degrades.  
These are well below the model's 518 px training resolution and well below the
ViT patch size floor.

In [ ]:
from src.resolution_sweep import EXTENDED_RESOLUTIONS

sweeper_ext = ResolutionSweeper(
    resolutions      = EXTENDED_RESOLUTIONS,   # [112, 168, 224, 280, 336, 392, 448, 518]
    conf_thresh      = 5.0,
    store_extrinsics = False,
)
# Reuse already-loaded model weights
sweeper_ext._model  = sweeper._model
sweeper_ext._device = sweeper._device
sweeper_ext._dtype  = sweeper._dtype

results_ext = sweeper_ext.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = True,
    with_scale    = True,
)
df_ext = sweeper_ext.to_dataframe(results_ext)

print("\nExtended resolution sweep:")
print(df_ext[["resolution", "ate_mean", "ate_rmse", "rpe_trans", "rpe_rot",
               "rot_mean_deg", "time_s", "peak_mb"]].to_string(
    index=False, float_format="{:.4f}".format))

# --- Plot ---
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("ATE mean (m)", "Inference Time (s)"))

fig.add_trace(go.Scatter(
    x=df_ext["resolution"], y=df_ext["ate_mean"],
    mode="lines+markers", name="ATE mean",
    line=dict(color="royalblue", width=2), marker=dict(size=8)),
    row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_ext["resolution"], y=df_ext["time_s"],
    mode="lines+markers", name="Time (s)",
    line=dict(color="steelblue", width=2), marker=dict(size=8)),
    row=1, col=2)

for col in (1, 2):
    fig.add_vline(x=224, line_dash="dot", line_color="orange",
                  annotation_text="224 px", annotation_position="top right",
                  row=1, col=col)
    fig.add_vline(x=518, line_dash="dash", line_color="grey",
                  annotation_text="518 px", annotation_position="top left",
                  row=1, col=col)

fig.update_xaxes(title_text="Resolution (px)")
fig.update_layout(height=420,
                  title_text="Extended Sweep: 112–518 px (single-pass, 8 frames)",
                  showlegend=False)
fig.show()

df_ext.to_csv(f"results/phase5c_resolution_sweep_extended_{SEQUENCE}.csv", index=False)
print(f"\nSaved to results/phase5c_resolution_sweep_extended_{SEQUENCE}.csv")